# <font color= #FFD700><b>Desenvolvimento de uma API para aprendizado</b></font>

Esse projeto começou com a ideia de criar um raspara tela para entrar num site qualquer e fazer pesquisa de produtos e retornar uma tabela.<br>
Mas agora vamos tentar criar uma API para fazer isso (já que o raspa tela já funcionou no outro notebook)<br>
A primeira tentativa foi usar a API do MercadoLivre mas não funcionou.

API REST:<br>
- API = Application Programming Interface - É uma interface que permite que um sistema converse com outro via regras definidas.
- REST = Representational State Transfer - É um estilo arquitetural para construir APIs sobre HTTP.

Métodos principais:
| Método | Função       |
| ------ | ------------ |
| GET    | Buscar dados |
| POST   | Criar dados  |
| PUT    | Atualizar    |
| DELETE | Remover      |


1. Coleta via API
2. Normalização JSON
3. DataFrame pandas
4. Limpeza e tipagem
5. Análise exploratória
6. Métricas de negócio

# Desenvolvimento

Site:
https://fakestoreapi.com/

In [32]:
import requests
import pandas as pd
import json

In [17]:
BASE_URL = "https://fakestoreapi.com/products"

response = requests.get(BASE_URL)

print("Status:", response.status_code)

data = response.json()
print("Quantidade de produtos:", len(data))

Status: 200
Quantidade de produtos: 20


In [ ]:
data[0] #exemplo de um produto

{'id': 1,
 'title': 'Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops',
 'price': 109.95,
 'description': 'Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday',
 'category': "men's clothing",
 'image': 'https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png',
 'rating': {'rate': 3.9, 'count': 120}}

Normalizando o JSON

In [23]:
df = pd.json_normalize(data)

df = df.rename(columns={
    "rating.rate": "rating",
    "rating.count": "rating_count"
})

df.head()

,id,title,price,description,category,image,rating,rating_count
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,Your perfect pack for everyday use and walks i...,men's clothing,https://fakestoreapi.com/img/81fPKd-2AYL._AC_S...,3.9,120
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,"Slim-fitting style, contrast raglan long sleev...",men's clothing,https://fakestoreapi.com/img/71-3HjGNDUL._AC_S...,4.1,259
2,3,Mens Cotton Jacket,55.99,great outerwear jackets for Spring/Autumn/Wint...,men's clothing,https://fakestoreapi.com/img/71li-ujtlUL._AC_U...,4.7,500
3,4,Mens Casual Slim Fit,15.99,The color could be slightly different between ...,men's clothing,https://fakestoreapi.com/img/71YXzeOuslL._AC_U...,2.1,430
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,"From our Legends Collection, the Naga was insp...",jewelery,https://fakestoreapi.com/img/71pWzhdJNwL._AC_U...,4.6,400


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            20 non-null     int64  
 1   title         20 non-null     str    
 2   price         20 non-null     float64
 3   description   20 non-null     str    
 4   category      20 non-null     str    
 5   image         20 non-null     str    
 6   rating        20 non-null     float64
 7   rating_count  20 non-null     int64  
dtypes: float64(2), int64(2), str(4)
memory usage: 1.4 KB


In [ ]:
# Com tratamento de erros

def fetch_products():
    try: # tratamento de exceções
        r = requests.get(BASE_URL, timeout=10) # envia uma requisição GET para a API com timeout de 10 segundos
        r.raise_for_status() #verifica se a resposta foi bem-sucedida (código 200), caso contrário, levanta uma exceção HTTPError (400, 404, 500, etc.)
        return r.json() # O retorno é um objeto response, aqui somente o conteúdo JSON é retornado, que é a lista de produtos
    except requests.exceptions.RequestException as e: #captura qualquer erro relacionado à requisição, como problemas de conexão, timeout ou erros HTTP
        print("Erro na requisição:", e)
        return [] #retorna uma lista vazia em caso de erro, para evitar que o programa quebre ao tentar processar os dados (precisa ser do mesmo tipo do retorno esperado)

In [31]:
print(r.headers)

{'Date': 'Sun, 01 Mar 2026 20:40:40 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'access-control-allow-origin': '*', 'etag': 'W/"2990-lZ2o94A2EDKiYW3/6/3oI74HHkw"', 'x-powered-by': 'Express', 'cf-cache-status': 'DYNAMIC', 'Nel': '{"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}', 'Report-To': '{"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=IfkxX8yXxYub5N3WxVQS1U8136b0sfLRC8oMpgppMvPKZIyGgX1UaVfHrcWcwr6nq7QeGt7dRfzdiD809h68V7DapBNj%2FkfQ6p4239i3N2I%3D"}]}', 'Content-Encoding': 'gzip', 'Server': 'cloudflare', 'CF-RAY': '9d5afa5f8e15a529-GRU', 'alt-svc': 'h3=":443"; ma=86400'}


Objeto response - a resposta HTTP completa do servidor: 
- Código de status
- Headers
- Corpo da resposta
- Informações de encoding
- Histórico de redirecionamentos
- Metadados da requisição

In [ ]:
df = pd.json_normalize(fetch_products())

In [50]:
df.head()

,id,title,price,description,category,image,rating.rate,rating.count
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,Your perfect pack for everyday use and walks i...,men's clothing,https://fakestoreapi.com/img/81fPKd-2AYL._AC_S...,3.9,120
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,"Slim-fitting style, contrast raglan long sleev...",men's clothing,https://fakestoreapi.com/img/71-3HjGNDUL._AC_S...,4.1,259
2,3,Mens Cotton Jacket,55.99,great outerwear jackets for Spring/Autumn/Wint...,men's clothing,https://fakestoreapi.com/img/71li-ujtlUL._AC_U...,4.7,500
3,4,Mens Casual Slim Fit,15.99,The color could be slightly different between ...,men's clothing,https://fakestoreapi.com/img/71YXzeOuslL._AC_U...,2.1,430
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,"From our Legends Collection, the Naga was insp...",jewelery,https://fakestoreapi.com/img/71pWzhdJNwL._AC_U...,4.6,400


## Mocking de Dependência Externa
Para simular (mockar) o retorno de uma pesquisa, para não trazer informações inúteis e reduzir o tamanho do df.

In [ ]:
# faz uma única requisição e salva o Json - resposta salva localmente (cache)

url = "https://fakestoreapi.com/products"

r = requests.get(url)
data = r.json()

with open("products_cache.json", "w") as f:
    json.dump(data, f)

In [35]:
# para simular

def fetch_products_mock():
    with open("products_cache.json", "r") as f:
        return json.load(f)

In [41]:
def search_products_local(query):
    data = fetch_products_mock()
    query = query.lower()
    
    filtered = [
        item for item in data
        if query in item["title"].lower()
    ]
    
    return filtered

In [51]:
results = search_products_local("pack")

In [52]:
results

[{'id': 1,
  'title': 'Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops',
  'price': 109.95,
  'description': 'Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday',
  'category': "men's clothing",
  'image': 'https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png',
  'rating': {'rate': 3.9, 'count': 120}}]